# Web search + text chunking

This notebook follows the same workflow as the grading-system demos: **environment setup**, then **numbered examples**—first chunking strategies alone (as in `001_chunking.ipynb`), then the **built-in web search** tool with optional **post-processing** of long answers via `utils.chunking`.

- **Web search**: `utils/web_search.py` (server-side `web_search_20250305`) — [docs](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/web-search-tool)
- **Chunking**: `utils/chunking.py` (character, sentence, section strategies)
- **Prompts**: `prompts/web_search_prompts.py`

In [ ]:
import os
import sys

# Resolve repo root (folder that contains `utils/`), works from notebooks/ or notebooks/tool_use/
_repo = os.getcwd()
while _repo and not os.path.isdir(os.path.join(_repo, "utils")):
    _parent = os.path.dirname(_repo)
    if _parent == _repo:
        raise RuntimeError("Could not find project root (directory containing utils/)")
    _repo = _parent
sys.path.insert(0, _repo)

from dotenv import load_dotenv

load_dotenv()

## Example 1 — Chunking strategies (static document)

Replicates the logic from `001_chunking.ipynb`: **character** sliding windows, **sentence** windows, and **section** splits on Markdown `##` headings. Implementations are in `utils.chunking` as `CharacterChunker`, `SentenceChunker`, and `SectionChunker`.

In [ ]:
from utils.chunking import CharacterChunker, SectionChunker, SentenceChunker, chunk_text

sample_report = """# Quarterly summary

## Revenue
Revenue increased. Margins held steady.

## Risks
Supply chain remains a concern. FX exposure is monitored.

End of report.
"""

char_chunks = chunk_text(sample_report, CharacterChunker(chunk_size=80, chunk_overlap=10))
sent_chunks = chunk_text(sample_report, SentenceChunker(max_sentences_per_chunk=2, overlap_sentences=1))
sec_chunks = chunk_text(sample_report, SectionChunker())

print("--- Character chunks:", len(char_chunks))
for i, c in enumerate(char_chunks[:3]):
    print(f"[{i}]", repr(c[:60]) + ("..." if len(c) > 60 else ""))
print("--- Sentence chunks:", len(sent_chunks))
print("--- Section chunks:", len(sec_chunks))
for i, s in enumerate(sec_chunks):
    print(f"section[{i}] head:", s[:70].replace("\n", " ") + "...")

## Example 2 — Initialize web search client

Restrict searches to **`nih.gov`** for evidence-aligned health queries (same pattern as `006_web_search.ipynb`).

In [ ]:
from prompts.web_search_prompts import (
    SCHOLAR_FINANCE_RESEARCH_SYSTEM,
    WEB_SEARCH_CHUNK_FOLLOWUP_SYSTEM,
    WEB_SEARCH_SYSTEM_GROUNDED,
    scholar_latest_financial_models_user,
)
from utils.web_search import WebSearchClient, WebSearchToolConfig, text_from_message

api_key = os.getenv("ANTHROPIC_API_KEY")
if api_key is None:
    raise ValueError("ANTHROPIC_API_KEY not found — add it to a .env file in the repo root.")

model = os.getenv("ANTHROPIC_WEB_SEARCH_MODEL", "claude-sonnet-4-5")

web_search_config = WebSearchToolConfig(
    max_uses=5,
    allowed_domains=["nih.gov"],
)

client = WebSearchClient(
    api_key=api_key,
    model=model,
    default_tool_config=web_search_config,
)

## Example 3 — Web-grounded answer, then chunk for downstream use

After `text_from_message`, split the assistant text (e.g. for per-chunk summarization, embedding, or storage). Adjust `CharacterChunker` sizes to your pipeline.

In [ ]:
user_prompt = """
What are evidence-based tips for leg strength training?
""".strip()

response = client.ask(
    user_prompt,
    system=WEB_SEARCH_SYSTEM_GROUNDED,
    max_tokens=1500,
    temperature=1.0,
)

answer_text = text_from_message(response)
print(answer_text[:800], "\n...\n" if len(answer_text) > 800 else "")

chunks_for_pipeline = chunk_text(
    answer_text,
    CharacterChunker(chunk_size=400, chunk_overlap=40),
)
print("Chunk count:", len(chunks_for_pipeline))
print("Chunk lengths:", [len(c) for c in chunks_for_pipeline[:5]])

## Example 4 — Google Scholar override

Override `tool_config` for a single call to restrict to **`scholar.google.com`** (finance research prompt).

In [ ]:
scholar_tool_config = WebSearchToolConfig(
    max_uses=5,
    allowed_domains=["scholar.google.com"],
)

response_scholar = client.ask(
    scholar_latest_financial_models_user(),
    system=SCHOLAR_FINANCE_RESEARCH_SYSTEM,
    tool_config=scholar_tool_config,
    max_tokens=2000,
    temperature=0.4,
)

scholar_text = text_from_message(response_scholar)
scholar_sentence_chunks = chunk_text(scholar_text, SentenceChunker(max_sentences_per_chunk=4, overlap_sentences=1))
print("Scholar answer (preview):\n", scholar_text[:600], "..." if len(scholar_text) > 600 else "")
print("Sentence chunks:", len(scholar_sentence_chunks))

## Optional — chunk-aware follow-up prompt

If you send chunks to the model in a second pass, use `WEB_SEARCH_CHUNK_FOLLOWUP_SYSTEM` so the model treats each block as a partial view of a longer answer.

In [ ]:
# Example: first chunk only (no extra API call required for illustration)
if chunks_for_pipeline:
    print(WEB_SEARCH_CHUNK_FOLLOWUP_SYSTEM)
    print("--- first chunk preview ---")
    print(chunks_for_pipeline[0][:500])